In [39]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [40]:
from langsmith import traceable
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openrouter import ChatOpenRouter
from langgraph.graph import START, StateGraph, MessagesState
from langgraph.prebuilt import tools_condition, ToolNode

In [41]:
api_key=os.getenv("OPENROUTER_API_KEY")

### Ignore Warnings

In [42]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="Pydantic serializer warnings:",
    category=UserWarning,
)

### LLM

In [43]:
llm = ChatOpenRouter(
    model="nvidia/nemotron-3-ultra-550b-a55b:free",
    api_key=api_key,
)

### Tools

In [44]:
@traceable(run_type="tool")
def add(a: int, b: int) -> int:
    """Performs Addition of a and b
    
    Arguments:
        a: first int
        b: second int
    """ 
    return a + b

def sub(a: int, b: int) -> int:
    """Performs Substraction of a and b
    
    Arguments:
        a: first int
        b: second int
    """ 
    return a - b

def multiply(a: int, b: int) -> int:
    """Performs multiplication of a and b
    
    Arguments:
        a: first int
        b: second int
    """ 
    return a * b


def div(a: int, b: int) -> int:
    """Performs division of a and b
    
    Arguments:
        a: first int
        b: second int
    """ 
    return a / b

In [45]:
tools = [add, sub, multiply, div]

### Bind LLM with Tools

In [46]:
llm_with_tools = llm.bind_tools(tools)

### System Message

In [47]:
system_msg = SystemMessage(
    content="""You are a helpful assistant.
    Your task is to perform suitable arithmetic operations on the given inputs."""
)

### Assistant

In [48]:
@traceable
def assistant(state: MessagesState):
    result = {"messages": [llm_with_tools.invoke([system_msg] + state["messages"])]}
    return result

### Graph

In [49]:
graph = StateGraph(MessagesState)

graph.add_node("assistant", assistant)
graph.add_node("tools", ToolNode(tools))

graph.add_edge(START, "assistant")
graph.add_conditional_edges(
    "assistant",
    tools_condition,
)
graph.add_edge("tools", "assistant")

app = graph.compile()

In [50]:
input_message = HumanMessage(content="hi! Could you please perform the addition of two numbers i.e 34 and 5")
result = app.invoke({"messages": [input_message]})

for m in result['messages'][-1:]:
    m.pretty_print()

================================== Ai Message ==================================

The addition of 34 and 5 is **39**.
